<a href="https://colab.research.google.com/github/marcos-mansur/load-forecast/blob/main/Data_quality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Objective

Verify data quality
- identify missing days
- input with day before

# load libs

In [ ]:
!pip install pendulum

In [6]:
import tensorflow as tf
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.utils import timeseries_dataset_from_array
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pendulum
#import optuna

plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['axes.grid'] = False

#load data 

In [174]:
df_20XX = pd.read_csv(f'https://ons-dl-prod-opendata.s3.amazonaws.com/dataset/carga_energia_di/CARGA_ENERGIA_2001.csv', 
                      sep=';', 
                      parse_dates=['din_instante'])

for x in range(2002,2022):
  df_20XX = pd.concat(objs = (df_20XX,pd.read_csv(f'https://ons-dl-prod-opendata.s3.amazonaws.com/dataset/carga_energia_di/CARGA_ENERGIA_{x}.csv', 
                      sep=';', 
                      parse_dates=['din_instante'])))

load_col = 'val_cargaenergiamwmed'
time_col = 'din_instante'

# target

In [175]:
def create_target_df(df):
  if df['din_instante'].iloc[0].day_name() != 'Friday':
    # get next friday - begins the operative week
    next_fri = get_friday(df['din_instante'].iloc[0])
    # df starts with the begin of operative week
    df = df[df['din_instante'] >= next_fri].copy()
  # average daily load by operative week
  df_target = pd.DataFrame(data=df.groupby(by=['semana'])[load_col].mean())
  # start day of each operative week
  df_target['Data'] = df.groupby(by=['semana'])[time_col].min()
  return df_target

df_target = create_target_df(df)

# treat data

In [236]:
def get_friday(date_time): 
  """ get next friday = start the operative week"""
  
  # today
  dt = pendulum.datetime(date_time.year,date_time.month, date_time.day)
  # return next friday
  return  dt.next(pendulum.FRIDAY).strftime('%Y-%m-%d')


def treat_data(df,regiao='SUDESTE',operative_week_start=2):
  
  # round the values of load
  df['val_cargaenergiamwmed'] = np.round(df['val_cargaenergiamwmed'],2)
  # drop last 4 rows that doesn't have load values
  df.dropna(axis=0, how='any',inplace=True)
  # filter data by subsystem 
  try:
    df = df[df['nom_subsistema']==regiao].reset_index().drop('index',
                                                             axis=1).copy()
  except:
    pass
  # dropa colunas sobre região
  df.drop(labels=['nom_subsistema','id_subsistema'], 
          inplace=True, axis=1,errors='ignore')

  # if df['din_instante'].iloc[0].day_name() != 'Friday':
  #   # get next friday - begins the operative week
  #   next_fri = get_friday(df['din_instante'].iloc[0])
  #   # df starts with the begin of operative week
  #   df = df[df['din_instante'] >= next_fri].copy()

  # create column with week number 
  df.reset_index(inplace=True,drop=True)
  df['semana'] = (df.index)//7 

  df['Mes'] = df['din_instante'].dt.month
  df['dia semana'] = df['din_instante'].dt.day_name()
  df['dia mes'] = df['din_instante'].dt.day
  df['ano'] = df['din_instante'].dt.year
  
  return df

df = treat_data(df_20XX, regiao='SUDESTE')
df.head(3)

,din_instante,val_cargaenergiamwmed,semana,Mes,dia semana,dia mes,ano
0,2001-01-01,19729.23,0,1,Monday,1,2001
1,2001-01-02,24596.20,0,1,Tuesday,2,2001
2,2001-01-03,26063.22,0,1,Wednesday,3,2001


# check missing data

In [237]:
df.groupby(['ano'])[time_col].count()

ano
2001    365
2002    365
2003    365
2004    366
2005    365
2006    365
2007    365
2008    366
2009    365
2010    365
2011    365
2012    366
2013    365
2014    364
2015    365
2016    357
2017    365
2018    365
2019    365
2020    366
2021    365
Name: din_instante, dtype: int64

There's one missing day in 2014 and 9 in 2016 (leap year).

In [228]:
# range of every day from 2001 to 2021
time_delta = pd.date_range(start = '2001-01-01', end= '2021-12-31',freq='D')
# turn into df
df_time = pd.DataFrame(data={'data':time_delta})
# left join range of data with datas from dataset, missing days will become NaN
df_missing = df_time.join(df.set_index('din_instante'), on='data', how='left')
# missing days indexes
df_missing.Mes[df_missing.Mes.isnull()]
df_missing.loc[df_missing.Mes[df_missing.Mes.isnull()].index].data

4779   2014-02-01
5573   2016-04-05
5574   2016-04-06
5575   2016-04-07
5576   2016-04-08
5577   2016-04-09
5578   2016-04-10
5579   2016-04-11
5580   2016-04-12
5581   2016-04-13
Name: data, dtype: datetime64[ns]